# Stock Rating System Demo

- 支持单标的评分、批量评分、指标明细查看、Gamma 外部输入和正股看涨筛选。
- 注意：Gamma 数据需要手动从外部来源填入；不填时系统会进入说明模式。

## 1. 导入依赖

In [1]:
from __future__ import annotations

import json
from typing import Any

import pandas as pd

from src import get_bulk_stock_ratings, get_stock_rating
from src.api import StockRatingService
from src.utils import indicators, summary

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

## 2. 配置标的与 Gamma 输入

`gamma_inputs_by_ticker` 的键使用大写股票代码。没有 Gamma 数据的标的可以留空，系统会把 Gamma 指标标记为不可用，并在 `warnings` / `unavailable_indicators` 中说明原因。

In [2]:
# 单标的分析对象
TICKER = "GOOG"

# 批量评分对象
TICKERS = ["GOOG", "AAPL", "MSFT", "NVDA", "TSLA"]

# Gamma 数据需从外部来源手动录入，例如 barchart 的 Gamma Exposure 页面。
# 字段名保持 README 中约定的 gex_net / gex_regime / gamma_wall / zero_gamma。
gamma_inputs_by_ticker: dict[str, dict[str, Any]] = {
    "GOOG": {
        "gex_net": 4000000,
        "gex_regime": "正Gamma",
        "gamma_wall": 340.00,
        "zero_gamma": 301.24,
        "source": "manual",
    },
    # "AAPL": {
    #     "gex_net": 1_250_000,
    #     "gex_regime": "正Gamma",
    #     "gamma_wall": 278,
    #     "zero_gamma": 265,
    #     "source": "manual",
    # },
}

single_gamma_inputs = gamma_inputs_by_ticker.get(TICKER.upper())

## 3. 展示辅助函数

In [3]:
def compact_rating_table(results: list[dict[str, Any]]) -> pd.DataFrame:
    """把批量评分结果压成适合排序浏览的表。"""
    rows = []
    for item in results:
        if "error" in item:
            rows.append({"ticker": item.get("ticker"), "error": item.get("error")})
            continue

        bull = item.get("bull_screening", {}) or {}
        rows.append(
            {
                "ticker": item.get("ticker"),
                "as_of": item.get("as_of"),
                "total_score": item.get("total_score"),
                "option_score": item.get("option_score"),
                "spot_score": item.get("spot_score"),
                "resonance_bonus": item.get("resonance_bonus"),
                "overall_direction": item.get("overall_direction"),
                "resonance_label": item.get("resonance_label"),
                "bull_passed": bull.get("passed_filter"),
                "bull_score": bull.get("bull_score"),
                "tier": bull.get("tier"),
                "safety_margin": bull.get("safety_margin"),
                "excess_pct": bull.get("excess_pct"),
                "warnings_count": len(item.get("warnings", [])),
            }
        )
    return pd.DataFrame(rows)


def bull_screening_table(result: dict[str, Any]) -> pd.DataFrame:
    """展开正股看涨筛选的 10 个维度。"""
    bull = result.get("bull_screening", {}) or {}
    dimensions = bull.get("dimensions", {}) or {}
    rows = []
    for name, dim in dimensions.items():
        rows.append(
            {
                "name": name,
                "score": dim.get("score"),
                "signal": dim.get("signal"),
                "raw_value": dim.get("raw_value"),
                "explain": dim.get("explain"),
                "available": dim.get("available"),
            }
        )
    return pd.DataFrame(rows)


def print_rating_brief(result: dict[str, Any]) -> None:
    """打印单标的评分摘要。"""
    print(
        f"{result['ticker']} | 综合分 {result['total_score']} "
        f"= 期权 {result['option_score']} + 现货 {result['spot_score']} + 共振 {result['resonance_bonus']}"
    )
    print(f"方向: {result['overall_direction']} | 期权: {result['option_direction']} | 现货: {result['spot_direction']}")
    print(f"共振: {result.get('resonance_label') or '无'}")

    bull = result.get("bull_screening", {}) or {}
    if bull.get("passed_filter"):
        print(f"看涨筛选: 通过 | 看涨分 {bull.get('bull_score')} | 梯队 {bull.get('tier')}")
        print(f"安全边际: {bull.get('safety_margin'):.1%} | 突破 γ Wall: {bull.get('excess_pct'):.1%}")
    else:
        print("看涨筛选: 未通过")
        for reason in bull.get("filter_reasons", []):
            print(f"- {reason}")

    if result.get("warnings"):
        print("\nWarnings:")
        for warning in result["warnings"]:
            print(f"- {warning}")

## 4. 单标的评分

In [4]:
single = get_stock_rating(
    ticker=TICKER,
    gamma_inputs=single_gamma_inputs,
)

print_rating_brief(single)

GOOG | 综合分 4 = 期权 0 + 现货 4 + 共振 0
方向: 以现货为主: 强看涨 | 期权: 中性 | 现货: 强看涨
共振: 无
看涨筛选: 未通过
- 综合分(4) < 最低阈值(8)
- Delta方向(None) <= 0，聪明钱未看涨

Warnings:
- 期权到期日获取失败: Too Many Requests. Rate limited. Try after a while.
- 共振判定: 期权分或现货分未达到共振阈值(>=3)。


### 单标的摘要表

In [5]:
summary_df = summary(single)
summary_df

,ticker,as_of,option_score,spot_score,resonance_bonus,total_score,option_direction,spot_direction,overall_direction,resonance_label
0,GOOG,1970-01-01,0,4,0,4,中性,强看涨,以现货为主: 强看涨,


### 指标明细

In [6]:
indicators_df = indicators(single)
indicators_df.sort_values(["category", "score"], ascending=[True, False])

,ticker,as_of,category,name,raw_value,score,signal,explain,available
13,GOOG,1970-01-01,gamma_indicators,GEX净值,4000000.0,0,正Gamma,来自外部输入: manual,True
14,GOOG,1970-01-01,gamma_indicators,GEX环境,正Gamma,0,正Gamma,来自外部输入: manual,True
15,GOOG,1970-01-01,gamma_indicators,γ Wall,340.0,0,关键位,来自外部输入: manual,True
16,GOOG,1970-01-01,gamma_indicators,Zero Gamma,301.24,0,翻转线,来自外部输入: manual,True
0,GOOG,1970-01-01,option_indicators,Vol/OI极端,None,0,中性,期权链数据为空，无法计算该指标。,False
1,GOOG,1970-01-01,option_indicators,P/C Ratio,None,0,中性,期权链数据为空，无法计算该指标。,False
2,GOOG,1970-01-01,option_indicators,末日期权效应,None,0,中性,期权链数据为空，无法计算该指标。,False
3,GOOG,1970-01-01,option_indicators,IV Skew反转,None,0,中性,无法定位前月ATM的Call/Put IV。,False
4,GOOG,1970-01-01,option_indicators,Delta加权方向,None,0,中性,期权链数据为空，无法计算该指标。,False
5,GOOG,1970-01-01,option_indicators,IV/HV比率,"{'atm_iv': None, 'hv20': 0.3898, 'iv_hv': None}",0,中性,ATM IV或HV不足，IV/HV不可计算。,False


### 不可用指标与提示

In [7]:
pd.DataFrame(
    [{"name": name, "reason": reason} for name, reason in single.get("unavailable_indicators", {}).items()]
)

""


## 5. 正股看涨筛选

In [8]:
bull = single.get("bull_screening", {})
pd.DataFrame([bull_screening_table(single).sum(numeric_only=True).rename("score_sum")]) if bull.get("passed_filter") else pd.DataFrame({"filter_reasons": bull.get("filter_reasons", [])})

,filter_reasons
0,综合分(4) < 最低阈值(8)
1,Delta方向(None) <= 0，聪明钱未看涨


In [9]:
bull_screening_table(single)

""


## 6. 批量评分与排序

In [10]:
bulk_results = get_bulk_stock_ratings(
    tickers=TICKERS,
    gamma_inputs_by_ticker=gamma_inputs_by_ticker,
)

ranking_df = compact_rating_table(bulk_results)
ranking_df.sort_values("total_score", ascending=False, na_position="last")

KeyboardInterrupt: 

## 7. 指标开关示例

下面示例展示如何使用 `StockRatingService` 在运行时禁用/恢复指标。默认不执行真实评分，避免不小心触发重复拉取。

In [ ]:
svc = StockRatingService()

base_indicators = pd.DataFrame(svc.engine.list_indicators())
bull_indicators = pd.DataFrame(svc.bull_screener.engine.list_indicators())

display(base_indicators)
display(bull_indicators)

# 示例：禁用后再恢复
# svc.engine.disable("P/C Ratio", "布林带压缩")
# result_without_some_indicators = svc.get_stock_rating(TICKER, gamma_inputs=single_gamma_inputs)
# svc.engine.enable_all()

## 8. 查看完整原始结果

当需要排查字段或导出结果时，可以直接查看 JSON。

In [ ]:
print(json.dumps(single, ensure_ascii=False, indent=2, default=str))